# AlphaGenome Genomic Prediction

![AlphaGenome Genomic Prediction](https://proto-bio.github.io/proto-assets/images/tool/alphagenome/hero.png)

AlphaGenome is a multi-task genomic foundation model from Google DeepMind that predicts regulatory signals, splicing, and 3D contact maps from DNA sequence at single base-pair resolution. It supports multiple context lengths (16 Kb to 1 Mb) and provides six tool functions covering interval prediction, variant-effect prediction, raw sequence prediction, variant scoring, interval scoring, and in-silico saturating mutagenesis.

This notebook demonstrates four representative functions: interval prediction, variant-effect prediction, variant scoring, and in-silico mutagenesis. The remaining two functions (`run_alphagenome_predict_sequences` and `run_alphagenome_score_intervals`) follow the same patterns and share input schemas with the functions shown here.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("alphagenome")
display_overview("alphagenome")
display_docs_section("alphagenome", "Background")

# AlphaGenome

AlphaGenome is a [deep learning](https://en.wikipedia.org/wiki/Deep_learning) model for regulatory genomics developed by Google DeepMind. It predicts a broad range of functional genomic measurements directly from DNA sequence across context windows of up to roughly one million base pairs, spanning [gene expression](https://en.wikipedia.org/wiki/Gene_expression), [chromatin accessibility](https://en.wikipedia.org/wiki/ATAC-seq), [transcription factor](https://en.wikipedia.org/wiki/Transcription_factor) binding, histone modifications, [RNA splicing](https://en.wikipedia.org/wiki/RNA_splicing), and three-dimensional [chromatin contacts](https://en.wikipedia.org/wiki/Chromosome_conformation_capture). This tool implementation provides six operations covering interval, sequence, and variant prediction alongside three scoring modes.

Gene regulation is encoded in non-coding DNA through [cis-regulatory elements](https://en.wikipedia.org/wiki/Cis-regulatory_element) such as promoters, [enhancers](https://en.wikipedia.org/wiki/Enhancer_(genetics)), and insulators, whose activity depends on sequence context that can extend across hundreds of kilobases. Relating a DNA sequence, or a non-coding [genetic variant](https://en.wikipedia.org/wiki/Mutation), to its functional consequences therefore requires models that read long stretches of sequence and predict many regulatory readouts together.

AlphaGenome ([Avsec et al., 2026](https://doi.org/10.1038/s41586-025-10014-0)) is a sequence-to-function model that accepts a genomic interval of up to roughly one megabase and predicts thousands of genome tracks at base or near-base resolution. The predicted assays span [RNA-seq](https://en.wikipedia.org/wiki/RNA-Seq) coverage, [CAGE](https://en.wikipedia.org/wiki/Cap_analysis_of_gene_expression) and PRO-cap transcription initiation, [ATAC-seq](https://en.wikipedia.org/wiki/ATAC-seq) and [DNase-seq](https://en.wikipedia.org/wiki/DNase-Seq) chromatin accessibility, [ChIP-seq](https://en.wikipedia.org/wiki/ChIP_sequencing) profiles for histone modifications and transcription factors, splice site positions, splice site usage and junctions, and [chromatin contact maps](https://en.wikipedia.org/wiki/Chromosome_conformation_capture). Because the model scores arbitrary sequence, the effect of a variant can be estimated by comparing predictions for the reference and alternate alleles, which supports interpretation of non-coding variation and systematic [in silico mutagenesis](https://en.wikipedia.org/wiki/Saturation_mutagenesis) of regulatory regions. Models are available for both the human and mouse genomes.

## Available tools

In [2]:
display_available_tools("alphagenome")

- **`run_alphagenome_predict_intervals()`** — Predict genomic signals for batched intervals using AlphaGenome
- **`run_alphagenome_predict_sequences()`** — Predict genomic signals from batched raw DNA sequences using AlphaGenome
- **`run_alphagenome_predict_variants()`** — Predict variant effects in batch using AlphaGenome
- **`run_alphagenome_score_intervals()`** — Score genomic intervals in batch with AlphaGenome interval scorers
- **`run_alphagenome_score_ism_variants_batch()`** — Run batched in-silico mutagenesis with AlphaGenome variant scorers
- **`run_alphagenome_score_variants()`** — Score variant effects in batch with AlphaGenome variant scorers

### `run_alphagenome_predict_intervals`

Given a genomic interval (chromosome, start, end), this function runs the AlphaGenome model and returns multi-track regulatory predictions for that region. The model supports RNA-seq, ATAC, DNase, histone ChIP, transcription factor ChIP, splice site, and contact map outputs. The interval is automatically resized to the nearest supported context length if needed. Output tracks can be filtered by ontology term (cell line or tissue) to focus on relevant biological contexts.

In [3]:
from proto_tools import (
    AlphaGenomeInterval,
    AlphaGenomePredictIntervalsConfig,
    AlphaGenomePredictIntervalsInput,
    run_alphagenome_predict_intervals,
)

In [4]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_predict_intervals")

# Paper Figure 2a coordinates: chr19, 1 Mb window, HepG2 cell line
interval_inputs = AlphaGenomePredictIntervalsInput(
    intervals=AlphaGenomeInterval(
        chromosome="chr19",
        interval_start=10_587_331,
        interval_end=11_635_907,
    ),
)

**Input** — `AlphaGenomePredictIntervalsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>intervals</code> | <code>list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeInterval]</code> | required | Genomic intervals for prediction |

In [5]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_predict_intervals")

interval_config = AlphaGenomePredictIntervalsConfig(
    requested_outputs=["RNA_SEQ", "ATAC", "DNASE", "CHIP_HISTONE", "CHIP_TF"],
    ontology_terms=["EFO:0001187"],  # HepG2
    organism="human",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaGenomePredictConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run AlphaGenome inference on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>1800</code> | Maximum execution time in seconds (JAX compilation is slow on first run). None = no cap. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>model_version</code> | <code>str</code> | <code>'all_folds'</code> | AlphaGenome Hugging Face model version |
| <code>requested_outputs</code> | <code>list[Literal['ATAC', 'CAGE', 'DNASE', 'RNA_SEQ', 'CHIP_HISTONE', 'CHIP_TF', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS', 'CONTACT_MAPS', 'PROCAP']]</code> | <code>['RNA_SEQ']</code> | Output type names to request from AlphaGenome |
| <code>ontology_terms</code> | <code>list[str] &#124; None</code> | <code>None</code> | UBERON tissue/cell IDs to filter tracks (e.g. 'UBERON:0001167'); None = all |
| <code>organism</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Organism for AlphaGenome predictions |

In [6]:
# Run the interval prediction function
interval_result = run_alphagenome_predict_intervals(interval_inputs, interval_config)

Running run_alphagenome_predict_intervals [00:00]

JAX compilation cache for alphagenome enabled at /large_storage/hielab/user/proto_cache/proto_tool_envs/alphagenome_env/jax_cache


In [7]:
display_api_reference("alphagenome", "output", "run_alphagenome_predict_intervals")

output = interval_result[0]
print(f"Requested outputs: {output.requested_outputs}")
print(f"Result keys: {list(output.result.get('predictions', {}).keys())}")

**Output** — `AlphaGenomePredictIntervalsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomePredictOutput]</code> | required | Per-interval AlphaGenome prediction outputs |

Requested outputs: ['RNA_SEQ', 'ATAC', 'DNASE', 'CHIP_HISTONE', 'CHIP_TF']
Result keys: ['atac', 'cage', 'dnase', 'rna_seq', 'chip_histone', 'chip_tf', 'splice_sites', 'splice_site_usage', 'splice_junctions', 'contact_maps', 'procap']


### `run_alphagenome_predict_variants`

Given a variant (chromosome, context window, variant position, reference and alternate alleles), this function runs AlphaGenome twice — once for the reference sequence and once for the alternate — and returns the full track-level predictions for both alleles. This is the raw prediction function that gives access to the complete output tensor for each allele. It reproduces Figure 3c from the AlphaGenome paper: a G-to-C variant at chr21:46,126,238 in the COL6A2 gene that causes alternative splice junction formation.

In [8]:
from proto_tools import (
    AlphaGenomePredictVariantsConfig,
    AlphaGenomePredictVariantsInput,
    AlphaGenomeVariant,
    run_alphagenome_predict_variants,
)

In [9]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_predict_variants")

# Paper Figure 3c coordinates: chr21:46,126,238 G>C (COL6A2), 1 Mb context, Aorta tissue
variant_inputs = AlphaGenomePredictVariantsInput(
    variants=AlphaGenomeVariant(
        chromosome="chr21",
        interval_start=45_601_950,
        interval_end=46_650_526,
        variant_position=46_126_238,
        reference_bases="G",
        alternate_bases="C",
    ),
)

**Input** — `AlphaGenomePredictVariantsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>variants</code> | <code>list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeVariant]</code> | required | Variants (with intervals) for prediction |

In [10]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_predict_variants")

variant_config = AlphaGenomePredictVariantsConfig(
    requested_outputs=["RNA_SEQ", "SPLICE_SITE_USAGE"],
    ontology_terms=["UBERON:0001496"],  # Aorta
    organism="human",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaGenomePredictConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run AlphaGenome inference on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>1800</code> | Maximum execution time in seconds (JAX compilation is slow on first run). None = no cap. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>model_version</code> | <code>str</code> | <code>'all_folds'</code> | AlphaGenome Hugging Face model version |
| <code>requested_outputs</code> | <code>list[Literal['ATAC', 'CAGE', 'DNASE', 'RNA_SEQ', 'CHIP_HISTONE', 'CHIP_TF', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS', 'CONTACT_MAPS', 'PROCAP']]</code> | <code>['RNA_SEQ']</code> | Output type names to request from AlphaGenome |
| <code>ontology_terms</code> | <code>list[str] &#124; None</code> | <code>None</code> | UBERON tissue/cell IDs to filter tracks (e.g. 'UBERON:0001167'); None = all |
| <code>organism</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Organism for AlphaGenome predictions |

In [11]:
# Run the variant-effect prediction function
variant_result = run_alphagenome_predict_variants(variant_inputs, variant_config)

Running run_alphagenome_predict_variants [00:00]

JAX compilation cache for alphagenome enabled at /large_storage/hielab/user/proto_cache/proto_tool_envs/alphagenome_env/jax_cache


In [12]:
display_api_reference("alphagenome", "output", "run_alphagenome_predict_variants")

output = variant_result[0]
print(f"Variant metadata: {output.variant}")
print(f"Result keys: {list(output.result.get('predictions', {}).keys())}")

**Output** — `AlphaGenomePredictVariantsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomePredictOutput]</code> | required | Per-variant AlphaGenome prediction outputs |

Variant metadata: {'position': 46126238, 'reference_bases': 'G', 'alternate_bases': 'C'}
Result keys: ['reference', 'alternate']


### `run_alphagenome_score_variants`

This function evaluates variants using AlphaGenome's recommended variant scorers and returns a tidy table of effect-size scores with one row per scorer-track-gene combination. It is more convenient than raw variant-effect prediction when you need quantitative effect sizes rather than full track-level predictions. The same COL6A2 variant from the previous section is used here to demonstrate the scoring output format.

In [13]:
import pandas as pd
from proto_tools import (
    AlphaGenomeScoreVariantsConfig,
    AlphaGenomeScoreVariantsInput,
    run_alphagenome_score_variants,
)

In [14]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_score_variants")

score_variant_inputs = AlphaGenomeScoreVariantsInput(
    variants=AlphaGenomeVariant(
        chromosome="chr21",
        interval_start=45_601_950,
        interval_end=46_650_526,
        variant_position=46_126_238,
        reference_bases="G",
        alternate_bases="C",
    ),
)

**Input** — `AlphaGenomeScoreVariantsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>variants</code> | <code>list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeVariant]</code> | required | Variants (with intervals) for scoring |

In [15]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_score_variants")

score_variant_config = AlphaGenomeScoreVariantsConfig(
    variant_scorers=["RNA_SEQ", "SPLICE_SITE_USAGE"],
    organism="human",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaGenomeScoreVariantsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run AlphaGenome inference on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>model_version</code> | <code>str</code> | <code>'all_folds'</code> | AlphaGenome Hugging Face model version |
| <code>variant_scorers</code> | <code>list[Literal['ATAC', 'CONTACT_MAPS', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'PROCAP', 'RNA_SEQ', 'RNA_SEQ_ACTIVE', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS', 'POLYADENYLATION', 'ATAC_ACTIVE', 'DNASE_ACTIVE', 'CHIP_TF_ACTIVE', 'CHIP_HISTONE_ACTIVE', 'CAGE_ACTIVE', 'PROCAP_ACTIVE']] &#124; None</code> | <code>None</code> | Scorer names to use. None uses all recommended scorers. |
| <code>organism</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Organism for AlphaGenome predictions |

In [16]:
# Run the variant scoring function
score_variant_result = run_alphagenome_score_variants(score_variant_inputs, score_variant_config)

Running run_alphagenome_score_variants [00:00]

JAX compilation cache for alphagenome enabled at /large_storage/hielab/user/proto_cache/proto_tool_envs/alphagenome_env/jax_cache


In [17]:
display_api_reference("alphagenome", "output", "run_alphagenome_score_variants")

output = score_variant_result[0]
print(f"Returned {len(output.scores)} score records")
if output.scores:
    df = pd.DataFrame(output.scores)
    print(df.head(10).to_string())

**Output** — `AlphaGenomeScoreVariantsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeScoreOutput]</code> | required | Per-variant AlphaGenome score outputs |

Returned 15415 score records
                                                                                                                                                                                                                                                                                                                                                                                                                                                  variant_id                                                                                                                                 scored_interval          gene_id        gene_name gene_type gene_strand junction_Start junction_End output_type                               variant_scorer                     track_name track_strand         Assay title ontology_curie          biosample_name                 biosample_type biosample_life_stage gtex_tissue data_source endedness genetically_modified  raw_score  quantile_score
0  {

### `run_alphagenome_score_ism_variants_batch`

In-silico saturating mutagenesis (ISM) systematically mutates every position within a sub-interval to every possible alternate base and scores each substitution. The result is a comprehensive map of positional sensitivity that reveals which bases within a regulatory element are critical for its function. This example saturate-mutates a 10 bp window around the COL6A2 variant, scoring each single-nucleotide substitution with the splice sites scorer.

In [18]:
from proto_tools import (
    AlphaGenomeISM,
    AlphaGenomeScoreISMConfig,
    AlphaGenomeScoreISMInput,
    run_alphagenome_score_ism_variants_batch,
)

In [19]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_score_ism_variants_batch")

ism_inputs = AlphaGenomeScoreISMInput(
    requests=AlphaGenomeISM(
        chromosome="chr21",
        interval_start=45_601_950,
        interval_end=46_650_526,
        ism_interval_start=46_126_233,
        ism_interval_end=46_126_243,  # 10 bp window around variant
    ),
)

**Input** — `AlphaGenomeScoreISMInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>requests</code> | <code>list[proto_tools.tools.sequence_scoring.alphagenome.alphagenome_score_ism_variants_batch.AlphaGenomeISM]</code> | required | ISM requests to process |

In [20]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_score_ism_variants_batch")

ism_config = AlphaGenomeScoreISMConfig(
    variant_scorers=["SPLICE_SITE_USAGE"],
    organism="human",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaGenomeScoreVariantsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run AlphaGenome inference on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>model_version</code> | <code>str</code> | <code>'all_folds'</code> | AlphaGenome Hugging Face model version |
| <code>variant_scorers</code> | <code>list[Literal['ATAC', 'CONTACT_MAPS', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'PROCAP', 'RNA_SEQ', 'RNA_SEQ_ACTIVE', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS', 'POLYADENYLATION', 'ATAC_ACTIVE', 'DNASE_ACTIVE', 'CHIP_TF_ACTIVE', 'CHIP_HISTONE_ACTIVE', 'CAGE_ACTIVE', 'PROCAP_ACTIVE']] &#124; None</code> | <code>None</code> | Scorer names to use. None uses all recommended scorers. |
| <code>organism</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Organism for AlphaGenome predictions |

In [21]:
# Run the ISM scoring function
ism_result = run_alphagenome_score_ism_variants_batch(ism_inputs, ism_config)

Running run_alphagenome_score_ism_variants_batch [00:00]

JAX compilation cache for alphagenome enabled at /large_storage/hielab/user/proto_cache/proto_tool_envs/alphagenome_env/jax_cache


In [22]:
display_api_reference("alphagenome", "output", "run_alphagenome_score_ism_variants_batch")

output = ism_result[0]
print(f"Returned {len(output.scores)} score records")
if output.scores:
    df = pd.DataFrame(output.scores)
    print(df.head(10).to_string())

**Output** — `AlphaGenomeScoreISMOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeScoreOutput]</code> | required | Per-request AlphaGenome ISM score outputs |

Returned 11010 score records
                                                                                                                                                                                                                                                                                                                                                                                                                                                  variant_id                                                                                                                                 scored_interval          gene_id gene_name       gene_type gene_strand junction_Start junction_End        output_type                                                          variant_scorer                           track_name track_strand         Assay title ontology_curie         biosample_name                 biosample_type biosample_life_stage gtex_tissue data_source  raw_score  quantile_sco

## Export Results

Prediction and scoring outputs can be saved to standard file formats for downstream analysis. Prediction outputs support JSON and NumPy formats, while scoring outputs support JSON and CSV. The export code below is shown for reference but not executed, as outputs can be very large.

In [23]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

interval_result[0].export("alphagenome_predict_intervals", export_path=output_dir, file_format="json")
variant_result[0].export("alphagenome_predict_variants", export_path=output_dir, file_format="json")
score_variant_result[0].export("alphagenome_score_variants", export_path=output_dir, file_format="csv")
ism_result[0].export("alphagenome_score_ism_variants_batch", export_path=output_dir, file_format="csv")